In [1]:
import os
import json
import math
import random
import warnings
from pathlib import Path
from collections import Counter, defaultdict
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
# Reproducibility
SEED = 23022006
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA devices:", torch.cuda.device_count())

Device: cuda
GPU: Tesla T4
CUDA devices: 2


In [3]:
# Config
MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

MAX_LENGTH = 192
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
EPOCHS = 8
VALID_RATIO = 0.1

LR_BERT = 2e-5
LR_HEAD = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
BEST_MODEL_PATH = OUTPUT_DIR / "biomedbert_bilstm_crf_ddi_best.pt"
LABEL_PATH = OUTPUT_DIR / "ddi_label_mapping.json"

In [4]:
def find_ddi_root():
    candidates = [
        Path("/kaggle/input/datasets/tuantc2306/dataset/DDICorpus"),
        Path("/kaggle/input/dataset/DDICorpus"),
        Path("/kaggle/input/datasets/DDICorpus"),
        Path("/kaggle/input/DDICorpus"),
        Path("dataset/DDICorpus"),
        Path("DDICorpus"),
    ]
    for path in candidates:
        if path.exists():
            return path

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        matches = list(kaggle_input.rglob("DDICorpus"))
        if matches:
            return matches[0]

    raise FileNotFoundError("Khong tim thay folder DDICorpus. Hay kiem tra lai Kaggle input path.")

DDI_ROOT = find_ddi_root()
TRAIN_DIR = DDI_ROOT / "Train"
TEST_ROOT = DDI_ROOT / "Test"
TEST_DRUG_NER_DIR = TEST_ROOT / "Test for DrugNER task"
TEST_DIR = TEST_DRUG_NER_DIR if TEST_DRUG_NER_DIR.exists() else TEST_ROOT

print("DDI_ROOT:", DDI_ROOT)
print("TRAIN_DIR:", TRAIN_DIR, "| exists:", TRAIN_DIR.exists())
print("TEST_DIR:", TEST_DIR, "| exists:", TEST_DIR.exists())
print("Train XML files:", len(list(TRAIN_DIR.rglob("*.xml"))))
print("Test XML files:", len(list(TEST_DIR.rglob("*.xml"))) if TEST_DIR.exists() else 0)

DDI_ROOT: /kaggle/input/datasets/tuantc2306/dataset/DDICorpus
TRAIN_DIR: /kaggle/input/datasets/tuantc2306/dataset/DDICorpus/Train | exists: True
TEST_DIR: /kaggle/input/datasets/tuantc2306/dataset/DDICorpus/Test/Test for DrugNER task | exists: True
Train XML files: 714
Test XML files: 112


In [5]:
def parse_char_offsets(offset_text):
    """DDI offsets are inclusive, sometimes discontinuous: '0-5;10-14'."""
    spans = []
    if not offset_text:
        return spans
    for part in offset_text.split(";"):
        part = part.strip()
        if not part or "-" not in part:
            continue
        start, end = part.split("-", 1)
        try:
            start, end = int(start), int(end)
        except ValueError:
            continue
        if end >= start:
            spans.append((start, end + 1))
    return spans


def read_xml_file(path):
    try:
        root = ET.parse(path).getroot()
    except ET.ParseError:
        text = Path(path).read_text(encoding="utf-8", errors="ignore")
        root = ET.fromstring(text)

    examples = []
    for sent in root.iter("sentence"):
        text = sent.attrib.get("text", "")
        if not text:
            continue

        entities = []
        for ent in sent.findall("entity"):
            ent_type = ent.attrib.get("type", "drug")
            ent_text = ent.attrib.get("text", "")
            char_offset = ent.attrib.get("charOffset", "")
            for start, end in parse_char_offsets(char_offset):
                if 0 <= start < end <= len(text):
                    entities.append({
                        "start": start,
                        "end": end,
                        "type": ent_type,
                        "text": ent_text,
                    })

        entities = sorted(entities, key=lambda x: (x["start"], x["end"]))
        examples.append({"id": sent.attrib.get("id", ""), "text": text, "entities": entities})
    return examples


def load_ddi_examples(folder):
    xml_files = sorted(Path(folder).rglob("*.xml"))
    examples = []
    skipped = []
    for xml_file in tqdm(xml_files, desc=f"Reading {folder}"):
        try:
            examples.extend(read_xml_file(xml_file))
        except Exception as exc:
            skipped.append((str(xml_file), str(exc)))
    return examples, skipped

train_examples_all, skipped_train = load_ddi_examples(TRAIN_DIR)
test_examples, skipped_test = load_ddi_examples(TEST_DIR) if TEST_DIR.exists() else ([], [])

print("Train sentences:", len(train_examples_all), "| skipped XML:", len(skipped_train))
print("Test sentences:", len(test_examples), "| skipped XML:", len(skipped_test))
print("Example:", train_examples_all[0] if train_examples_all else None)

Reading /kaggle/input/datasets/tuantc2306/dataset/DDICorpus/Train:   0%|          | 0/714 [00:00<?, ?it/s]

Reading /kaggle/input/datasets/tuantc2306/dataset/DDICorpus/Test/Test for DrugNER task:   0%|          | 0/112…

Train sentences: 6905 | skipped XML: 0
Test sentences: 665 | skipped XML: 0
Example: {'id': 'DDI-DrugBank.d436.s0', 'text': 'No drug, nutritional supplement, food or herb interactions have yet been reported.', 'entities': []}


In [6]:
print(train_examples_all[67] if train_examples_all else None)

{'id': 'DDI-DrugBank.d353.s5', 'text': 'However, it has been established that acitretin interferes with the contraceptive effect of microdosed progestin minipill preparations.', 'entities': [{'start': 38, 'end': 47, 'type': 'drug', 'text': 'acitretin'}, {'start': 103, 'end': 112, 'type': 'drug', 'text': 'progestin'}]}


In [7]:
def entity_type_counts(examples):
    counter = Counter()
    sent_with_entity = 0
    for ex in examples:
        if ex["entities"]:
            sent_with_entity += 1
        for ent in ex["entities"]:
            counter[ent["type"]] += 1
    return counter, sent_with_entity

train_type_counts, train_sent_with_ent = entity_type_counts(train_examples_all)
test_type_counts, test_sent_with_ent = entity_type_counts(test_examples)

print("Train entity types:", train_type_counts)
print("Train sentences with entity:", train_sent_with_ent)
print("Test entity types:", test_type_counts)
print("Test sentences with entity:", test_sent_with_ent)

Train entity types: Counter({'drug': 9432, 'group': 3429, 'brand': 1437, 'drug_n': 505})
Train sentences with entity: 5560
Test entity types: Counter({'drug': 352, 'group': 156, 'drug_n': 121, 'brand': 59})
Test sentences with entity: 324


In [8]:
# Split train into train/validation by sentence.
rng = random.Random(SEED)
indices = list(range(len(train_examples_all)))
rng.shuffle(indices)
valid_size = max(1, int(len(indices) * VALID_RATIO))
valid_idx = set(indices[:valid_size])

train_examples = [ex for i, ex in enumerate(train_examples_all) if i not in valid_idx]
valid_examples = [ex for i, ex in enumerate(train_examples_all) if i in valid_idx]

entity_types = sorted({ent["type"] for ex in train_examples_all for ent in ex["entities"]})
labels = ["O"]
for ent_type in entity_types:
    labels.extend([f"B-{ent_type}", f"I-{ent_type}"])

label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}

print("Train:", len(train_examples), "| Valid:", len(valid_examples), "| Test:", len(test_examples))
print("Labels:", labels)

with open(LABEL_PATH, "w", encoding="utf-8") as f:
    json.dump({"label2id": label2id, "id2label": id2label, "model_name": MODEL_NAME}, f, indent=2)
print("Saved label mapping to", LABEL_PATH)

Train: 6215 | Valid: 690 | Test: 665
Labels: ['O', 'B-brand', 'I-brand', 'B-drug', 'I-drug', 'B-drug_n', 'I-drug_n', 'B-group', 'I-group']
Saved label mapping to /kaggle/working/ddi_label_mapping.json


In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print(type(tokenizer))
print("Vocab size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

<class 'transformers.models.bert.tokenization_bert.BertTokenizer'>
Vocab size: 30522


In [10]:
def align_labels_with_offsets(offsets, entities, label2id):
    token_labels = [label2id["O"] for _ in offsets]
    eval_mask = []

    for idx, (start, end) in enumerate(offsets):
        is_special = (start == 0 and end == 0)
        eval_mask.append(0 if is_special else 1)

    occupied = [False for _ in offsets]
    for ent in sorted(entities, key=lambda x: (x["start"], x["end"])):
        first_token = True
        ent_start, ent_end, ent_type = ent["start"], ent["end"], ent["type"]
        b_label = label2id.get(f"B-{ent_type}", label2id["O"])
        i_label = label2id.get(f"I-{ent_type}", label2id["O"])

        for i, (tok_start, tok_end) in enumerate(offsets):
            if tok_start == 0 and tok_end == 0:
                continue
            overlaps = tok_start < ent_end and tok_end > ent_start
            if overlaps and not occupied[i]:
                token_labels[i] = b_label if first_token else i_label
                occupied[i] = True
                first_token = False

    return token_labels, eval_mask


class DDINerDataset(Dataset):
    def __init__(self, examples, tokenizer, label2id, max_length=192):
        self.examples = examples
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        encoding = self.tokenizer(
            ex["text"],
            truncation=True,
            max_length=self.max_length,
            return_offsets_mapping=True,
            add_special_tokens=True,
        )
        offsets = encoding.pop("offset_mapping")
        labels, eval_mask = align_labels_with_offsets(offsets, ex["entities"], self.label2id)

        return {
            "input_ids": torch.tensor(encoding["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoding["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "eval_mask": torch.tensor(eval_mask, dtype=torch.long),
        }


def collate_batch(batch):
    max_len = max(item["input_ids"].size(0) for item in batch)
    pad_id = tokenizer.pad_token_id
    o_id = label2id["O"]

    out = defaultdict(list)
    for item in batch:
        length = item["input_ids"].size(0)
        pad_len = max_len - length
        out["input_ids"].append(torch.cat([item["input_ids"], torch.full((pad_len,), pad_id, dtype=torch.long)]))
        out["attention_mask"].append(torch.cat([item["attention_mask"], torch.zeros(pad_len, dtype=torch.long)]))
        out["labels"].append(torch.cat([item["labels"], torch.full((pad_len,), o_id, dtype=torch.long)]))
        out["eval_mask"].append(torch.cat([item["eval_mask"], torch.zeros(pad_len, dtype=torch.long)]))

    return {key: torch.stack(value) for key, value in out.items()}

train_ds = DDINerDataset(train_examples, tokenizer, label2id, MAX_LENGTH)
valid_ds = DDINerDataset(valid_examples, tokenizer, label2id, MAX_LENGTH)
test_ds = DDINerDataset(test_examples, tokenizer, label2id, MAX_LENGTH) if test_examples else None

train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, collate_fn=collate_batch, num_workers=2)
valid_loader = DataLoader(valid_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_batch, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_batch, num_workers=2) if test_ds else None

batch = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in batch.items()})
print("Decoded sample:", tokenizer.convert_ids_to_tokens(batch["input_ids"][0][:30]))
print("Label sample:", [id2label[i.item()] for i in batch["labels"][0][:30]])

{'input_ids': (8, 36), 'attention_mask': (8, 36), 'labels': (8, 36), 'eval_mask': (8, 36)}
Decoded sample: ['[CLS]', 'buprenorphine', 'is', 'metabolized', 'to', 'nor', '##bu', '##prenorphine', 'by', 'cytochrome', 'cyp', '3a', '##4', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
Label sample: ['O', 'B-drug', 'O', 'O', 'O', 'B-drug_n', 'I-drug_n', 'I-drug_n', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [11]:
class LinearChainCRF(nn.Module):
    def __init__(self, num_tags):
        super().__init__()
        self.num_tags = num_tags
        self.start_transitions = nn.Parameter(torch.empty(num_tags))
        self.end_transitions = nn.Parameter(torch.empty(num_tags))
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.uniform_(self.start_transitions, -0.1, 0.1)
        nn.init.uniform_(self.end_transitions, -0.1, 0.1)
        nn.init.uniform_(self.transitions, -0.1, 0.1)

    def forward(self, emissions, tags, mask):
        emissions = emissions.float()
        mask = mask.bool()
        log_denominator = self._compute_log_partition(emissions, mask)
        log_numerator = self._compute_score(emissions, tags, mask)
        return torch.mean(log_denominator - log_numerator)

    def _compute_score(self, emissions, tags, mask):
        batch_size, seq_len, _ = emissions.shape
        score = self.start_transitions[tags[:, 0]]
        score += emissions[torch.arange(batch_size, device=emissions.device), 0, tags[:, 0]]

        for t in range(1, seq_len):
            emit_score = emissions[torch.arange(batch_size, device=emissions.device), t, tags[:, t]]
            trans_score = self.transitions[tags[:, t - 1], tags[:, t]]
            score += (emit_score + trans_score) * mask[:, t]

        lengths = mask.long().sum(dim=1).clamp(min=1) - 1
        last_tags = tags[torch.arange(batch_size, device=emissions.device), lengths]
        score += self.end_transitions[last_tags]
        return score

    def _compute_log_partition(self, emissions, mask):
        score = self.start_transitions + emissions[:, 0]
        for t in range(1, emissions.size(1)):
            next_score = score.unsqueeze(2) + self.transitions.unsqueeze(0) + emissions[:, t].unsqueeze(1)
            next_score = torch.logsumexp(next_score, dim=1)
            score = torch.where(mask[:, t].unsqueeze(1), next_score, score)
        score += self.end_transitions
        return torch.logsumexp(score, dim=1)

    @torch.no_grad()
    def decode(self, emissions, mask):
        mask = mask.bool()
        batch_size, seq_len, _ = emissions.shape
        score = self.start_transitions + emissions[:, 0]
        history = []

        for t in range(1, seq_len):
            next_score = score.unsqueeze(2) + self.transitions.unsqueeze(0) + emissions[:, t].unsqueeze(1)
            best_score, best_tag = next_score.max(dim=1)
            score = torch.where(mask[:, t].unsqueeze(1), best_score, score)
            history.append(best_tag)

        score += self.end_transitions
        best_last_tags = score.argmax(dim=1)
        lengths = mask.long().sum(dim=1).clamp(min=1)

        best_paths = []
        for b in range(batch_size):
            seq_len_b = lengths[b].item()
            best_tag = best_last_tags[b].item()
            path = [best_tag]
            for hist in reversed(history[: seq_len_b - 1]):
                best_tag = hist[b, best_tag].item()
                path.append(best_tag)
            path.reverse()
            best_paths.append(path)
        return best_paths


class BertBiLstmCrf(nn.Module):
    def __init__(self, model_name, num_labels, lstm_hidden_size=256, lstm_layers=1, dropout=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=lstm_hidden_size,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.0 if lstm_layers == 1 else dropout,
        )
        self.classifier = nn.Linear(lstm_hidden_size * 2, num_labels)
        self.crf = LinearChainCRF(num_labels)

    def emissions(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)
        lstm_output, _ = self.lstm(sequence_output)
        lstm_output = self.dropout(lstm_output)
        return self.classifier(lstm_output)

    def forward(self, input_ids, attention_mask, labels=None):
        emissions = self.emissions(input_ids, attention_mask)
        if labels is not None:
            loss = self.crf(emissions, labels, attention_mask)
            return loss, emissions
        return emissions

    def decode(self, input_ids, attention_mask):
        emissions = self.emissions(input_ids, attention_mask)
        return self.crf.decode(emissions, attention_mask)

model = BertBiLstmCrf(MODEL_NAME, num_labels=len(labels)).to(device)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Trainable parameters: 111588204


In [12]:
def get_entities_from_bio(tags):
    entities = set()
    ent_type = None
    start = None

    for i, tag in enumerate(tags):
        if tag == "O" or tag is None:
            if ent_type is not None:
                entities.add((ent_type, start, i - 1))
                ent_type, start = None, None
            continue

        prefix, typ = tag.split("-", 1) if "-" in tag else ("O", None)
        if prefix == "B" or ent_type != typ:
            if ent_type is not None:
                entities.add((ent_type, start, i - 1))
            ent_type, start = typ, i
        elif prefix == "I":
            continue
        else:
            if ent_type is not None:
                entities.add((ent_type, start, i - 1))
                ent_type, start = None, None

    if ent_type is not None:
        entities.add((ent_type, start, len(tags) - 1))
    return entities


def evaluate(model, dataloader, id2label, desc="Evaluating"):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    token_correct = 0
    token_total = 0
    gold_total = 0
    pred_total = 0
    correct_total = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=desc):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_tensor = batch["labels"].to(device)
            eval_mask = batch["eval_mask"].to(device)

            loss, _ = model(input_ids, attention_mask, labels_tensor)
            paths = model.decode(input_ids, attention_mask)

            total_loss += loss.item()
            n_batches += 1

            for b, path in enumerate(paths):
                gold_tags = []
                pred_tags = []
                valid_positions = eval_mask[b].bool().cpu().tolist()
                gold_ids = labels_tensor[b].detach().cpu().tolist()

                for pos, keep in enumerate(valid_positions[:len(path)]):
                    if not keep:
                        continue
                    gold = id2label[gold_ids[pos]]
                    pred = id2label[path[pos]]
                    gold_tags.append(gold)
                    pred_tags.append(pred)
                    token_correct += int(gold == pred)
                    token_total += 1

                gold_entities = get_entities_from_bio(gold_tags)
                pred_entities = get_entities_from_bio(pred_tags)
                gold_total += len(gold_entities)
                pred_total += len(pred_entities)
                correct_total += len(gold_entities & pred_entities)

    precision = correct_total / pred_total if pred_total else 0.0
    recall = correct_total / gold_total if gold_total else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    token_acc = token_correct / token_total if token_total else 0.0

    return {
        "loss": total_loss / max(n_batches, 1),
        "token_acc": token_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "gold_entities": gold_total,
        "pred_entities": pred_total,
        "correct_entities": correct_total,
    }

In [13]:
bert_params = list(model.bert.parameters())
head_params = list(model.lstm.parameters()) + list(model.classifier.parameters()) + list(model.crf.parameters())

optimizer = AdamW(
    [
        {"params": bert_params, "lr": LR_BERT, "weight_decay": WEIGHT_DECAY},
        {"params": head_params, "lr": LR_HEAD, "weight_decay": WEIGHT_DECAY},
    ]
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
print("Total steps:", total_steps, "| Warmup steps:", warmup_steps)

Total steps: 6216 | Warmup steps: 621


In [14]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("trainng_ddi_model1")


In [15]:
import wandb

WANDB_PROJECT = "ddi-ner-biomedbert-bilstm-crf2"
WANDB_RUN_NAME = f"{MODEL_NAME.split('/')[-1]}-maxlen{MAX_LENGTH}-seed{SEED}"


def wandb_config():
    return {
        "model_name": MODEL_NAME,
        "architecture": "BiomedBERT + BiLSTM + CRF",
        "max_length": MAX_LENGTH,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "epochs": EPOCHS,
        "valid_ratio": VALID_RATIO,
        "lr_bert": LR_BERT,
        "lr_head": LR_HEAD,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "grad_clip": GRAD_CLIP,
        "seed": SEED,
        "num_labels": len(labels),
        "labels": labels,
        "train_sentences": len(train_examples),
        "valid_sentences": len(valid_examples),
        "test_sentences": len(test_examples),
    }


def ensure_wandb_run():
    if wandb.run is None:
        run = wandb.init(
            project=WANDB_PROJECT,
            name=WANDB_RUN_NAME,
            config=wandb_config(),
        )
        wandb.watch(model, log="gradients", log_freq=100)
        return run
    return wandb.run

wandb.login(key=secret_value_0)
wandb_run = ensure_wandb_run()


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: noct2306 (noct2306-uet) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [16]:
wandb_run = ensure_wandb_run()

best_valid_f1 = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")

    for step, batch in enumerate(progress, start=1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_tensor = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            loss, _ = model(input_ids, attention_mask, labels_tensor)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss += loss.item()
        progress.set_postfix(loss=f"{train_loss / step:.4f}")

    train_loss /= max(len(train_loader), 1)
    train_metrics = evaluate(model, train_loader, id2label, desc=f"Train eval epoch {epoch}")
    valid_metrics = evaluate(model, valid_loader, id2label, desc=f"Valid epoch {epoch}")

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"valid_{k}": v for k, v in valid_metrics.items()},
    }
    history.append(row)
    print(row)

    wandb.log(
        {
            "epoch": epoch,
            "train/loss": train_loss,
            "train/eval_loss": train_metrics["loss"],
            "train/token_acc": train_metrics["token_acc"],
            "train/precision": train_metrics["precision"],
            "train/recall": train_metrics["recall"],
            "train/f1": train_metrics["f1"],
            "train/gold_entities": train_metrics["gold_entities"],
            "train/pred_entities": train_metrics["pred_entities"],
            "train/correct_entities": train_metrics["correct_entities"],
            "dev/loss": valid_metrics["loss"],
            "dev/token_acc": valid_metrics["token_acc"],
            "dev/precision": valid_metrics["precision"],
            "dev/recall": valid_metrics["recall"],
            "dev/f1": valid_metrics["f1"],
            "dev/gold_entities": valid_metrics["gold_entities"],
            "dev/pred_entities": valid_metrics["pred_entities"],
            "dev/correct_entities": valid_metrics["correct_entities"],
            "lr/bert": optimizer.param_groups[0]["lr"],
            "lr/head": optimizer.param_groups[1]["lr"],
        },
        step=epoch,
    )

    if valid_metrics["f1"] > best_valid_f1:
        best_valid_f1 = valid_metrics["f1"]
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "label2id": label2id,
                "id2label": id2label,
                "model_name": MODEL_NAME,
                "max_length": MAX_LENGTH,
                "valid_metrics": valid_metrics,
            },
            BEST_MODEL_PATH,
        )
        wandb.run.summary["best_valid_f1"] = best_valid_f1
        wandb.run.summary["best_epoch"] = epoch
        wandb.run.summary["best_model_path"] = str(BEST_MODEL_PATH)
        print("Saved best model to", BEST_MODEL_PATH)

history_df = pd.DataFrame(history)
wandb.log({"history": wandb.Table(dataframe=history_df)})
wandb.finish()
history_df

Epoch 1/8:   0%|          | 0/777 [00:00<?, ?it/s]

Train eval epoch 1:   0%|          | 0/777 [00:00<?, ?it/s]

Valid epoch 1:   0%|          | 0/44 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 1.5935692463448916, 'train_token_acc': 0.9831565775454072, 'train_precision': 0.9206780158497212, 'train_recall': 0.9440933032355154, 'train_f1': 0.9322386507169923, 'train_gold_entities': 13290, 'train_pred_entities': 13628, 'train_correct_entities': 12547, 'valid_loss': 2.3703978007489983, 'valid_token_acc': 0.975921620724012, 'valid_precision': 0.8885224274406333, 'valid_recall': 0.9226027397260274, 'valid_f1': 0.905241935483871, 'valid_gold_entities': 1460, 'valid_pred_entities': 1516, 'valid_correct_entities': 1347}
Saved best model to /kaggle/working/biomedbert_bilstm_crf_ddi_best.pt


Epoch 2/8:   0%|          | 0/777 [00:00<?, ?it/s]

...

Train eval epoch 8:   0%|          | 0/777 [00:00<?, ?it/s]

Valid epoch 8:   0%|          | 0/44 [00:00<?, ?it/s]

{'epoch': 8, 'train_loss': 0.1636535078041409, 'train_token_acc': 0.9986929157271354, 'train_precision': 0.9934556942981796, 'train_recall': 0.9937547027840482, 'train_f1': 0.9936051760457418, 'train_gold_entities': 13290, 'train_pred_entities': 13294, 'train_correct_entities': 13207, 'valid_loss': 3.0038127939809454, 'valid_token_acc': 0.9846673308978191, 'valid_precision': 0.9414535666218035, 'valid_recall': 0.9582191780821918, 'valid_f1': 0.9497623896809233, 'valid_gold_entities': 1460, 'valid_pred_entities': 1486, 'valid_correct_entities': 1399}


dev/correct_entities,▁▅▅▆█▇██
dev/f1,▁▄▆▇████
dev/gold_entities,▁▁▁▁▁▁▁▁
dev/loss,▁▁▂▂▂▆▆█
dev/precision,▁▄▇█▇██▇
dev/pred_entities,██▂▁▅▂▄▄
dev/recall,▁▅▅▆█▇██
dev/token_acc,▁▃▆▆████
epoch,▁▂▃▄▅▆▇█
lr/bert,█▇▆▅▄▃▂▁
+10,...


,epoch,train_loss,train_token_acc,train_precision,train_recall,train_f1,train_gold_entities,train_pred_entities,train_correct_entities,valid_loss,valid_token_acc,valid_precision,valid_recall,valid_f1,valid_gold_entities,valid_pred_entities,valid_correct_entities
0,1,1.593569,0.983157,0.920678,0.944093,0.932239,13290,13628,12547,2.370398,0.975922,0.888522,0.922603,0.905242,1460,1516,1347
1,2,1.146584,0.989537,0.951353,0.971181,0.961165,13290,13567,12907,2.386662,0.978800,0.909571,0.943836,0.926387,1460,1515,1378
2,3,0.774282,0.994227,0.975349,0.976524,0.975936,13290,13306,12978,2.477868,0.982785,0.934148,0.942466,0.938288,1460,1473,1376
3,4,0.521204,0.995719,0.982283,0.980361,0.981321,13290,13264,13029,2.422152,0.982841,0.945280,0.946575,0.945927,1460,1462,1382
4,5,0.354718,0.996934,0.984997,0.988036,0.986514,13290,13331,13131,2.502398,0.984833,0.938338,0.958904,0.948509,1460,1492,1400
5,6,0.257359,0.997788,0.990879,0.989090,0.989983,13290,13266,13145,2.791866,0.984999,0.946332,0.954110,0.950205,1460,1472,1393
6,7,0.190478,0.998358,0.991724,0.991798,0.991761,13290,13291,13181,2.863184,0.985276,0.942684,0.957534,0.950051,1460,1483,1398
7,8,0.163654,0.998693,0.993456,0.993755,0.993605,13290,13294,13207,3.003813,0.984667,0.941454,0.958219,0.949762,1460,1486,1399


In [17]:
# Load best checkpoint before final evaluation.
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
print("Best validation metrics:", checkpoint["valid_metrics"])

if test_loader is not None and len(test_ds) > 0:
    test_metrics = evaluate(model, test_loader, id2label, desc="Test")
    print("Test metrics:")
    print(json.dumps(test_metrics, indent=2))
else:
    print("Khong co test_loader. Kiem tra lai TEST_DIR neu can danh gia test.")

Best validation metrics: {'loss': 2.7918661169030448, 'token_acc': 0.9849994464740396, 'precision': 0.9463315217391305, 'recall': 0.9541095890410959, 'f1': 0.9502046384720327, 'gold_entities': 1460, 'pred_entities': 1472, 'correct_entities': 1393}


Test:   0%|          | 0/42 [00:00<?, ?it/s]

Test metrics:
{
  "loss": 5.183805741014934,
  "token_acc": 0.9696743438056401,
  "precision": 0.7341597796143251,
  "recall": 0.7769679300291545,
  "f1": 0.754957507082153,
  "gold_entities": 686,
  "pred_entities": 726,
  "correct_entities": 533
}


In [18]:
# Evaluate the best model separately on DrugBank and MedLine test subsets.
def load_ddi_examples_from_files(xml_files, desc="Reading XML files"):
    examples = []
    skipped = []
    for xml_file in tqdm(sorted(xml_files), desc=desc):
        try:
            examples.extend(read_xml_file(xml_file))
        except Exception as exc:
            skipped.append((str(xml_file), str(exc)))
    return examples, skipped


def find_test_xml_files_by_domain(domain_name):
    domain_key = domain_name.lower()
    candidate_dirs = [
        TEST_DIR / domain_name,
        TEST_DIR / domain_name.lower(),
        TEST_DIR / domain_name.upper(),
        TEST_ROOT / "Test for DrugNER task" / domain_name,
        TEST_ROOT / "Test for DrugNER task" / domain_name.lower(),
        TEST_ROOT / domain_name,
        TEST_ROOT / domain_name.lower(),
    ]

    for candidate in candidate_dirs:
        if candidate.exists():
            xml_files = sorted(candidate.rglob("*.xml"))
            if xml_files:
                return xml_files

    # Fallback: use path/file names containing the domain keyword.
    xml_files = []
    for xml_file in TEST_DIR.rglob("*.xml"):
        path_text = str(xml_file).lower()
        if domain_key in path_text:
            xml_files.append(xml_file)
    return sorted(xml_files)


def evaluate_test_domain(domain_name):
    xml_files = find_test_xml_files_by_domain(domain_name)
    print(f"{domain_name} XML files:", len(xml_files))

    if not xml_files:
        print(f"Khong tim thay XML cho {domain_name}. Hay kiem tra cau truc folder trong TEST_DIR:", TEST_DIR)
        return None

    examples, skipped = load_ddi_examples_from_files(xml_files, desc=f"Reading {domain_name}")
    print(f"{domain_name} sentences:", len(examples), "| skipped XML:", len(skipped))

    dataset = DDINerDataset(examples, tokenizer, label2id, MAX_LENGTH)
    loader = DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_batch,
        num_workers=2,
    )
    metrics = evaluate(model, loader, id2label, desc=f"Test {domain_name}")
    metrics = {"subset": domain_name, "sentences": len(examples), "xml_files": len(xml_files), **metrics}
    print(f"{domain_name} metrics:")
    print(json.dumps(metrics, indent=2))
    return metrics

# Make sure the best checkpoint is loaded before subset evaluation.
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

subset_results = []
for subset_name in ["DrugBank", "MedLine"]:
    result = evaluate_test_domain(subset_name)
    if result is not None:
        subset_results.append(result)

subset_results_df = pd.DataFrame(subset_results)
subset_results_df

DrugBank XML files: 54


Reading DrugBank:   0%|          | 0/54 [00:00<?, ?it/s]

DrugBank sentences: 145 | skipped XML: 0


Test DrugBank:   0%|          | 0/10 [00:00<?, ?it/s]

DrugBank metrics:
{
  "subset": "DrugBank",
  "sentences": 145,
  "xml_files": 54,
  "loss": 4.878257203102112,
  "token_acc": 0.9718232044198895,
  "precision": 0.8910256410256411,
  "recall": 0.9144736842105263,
  "f1": 0.9025974025974026,
  "gold_entities": 304,
  "pred_entities": 312,
  "correct_entities": 278
}
MedLine XML files: 58


Reading MedLine:   0%|          | 0/58 [00:00<?, ?it/s]

MedLine sentences: 520 | skipped XML: 0


Test MedLine:   0%|          | 0/33 [00:00<?, ?it/s]

MedLine metrics:
{
  "subset": "MedLine",
  "sentences": 520,
  "xml_files": 58,
  "loss": 5.398598546331579,
  "token_acc": 0.9691102893191211,
  "precision": 0.6159420289855072,
  "recall": 0.6675392670157068,
  "f1": 0.6407035175879398,
  "gold_entities": 382,
  "pred_entities": 414,
  "correct_entities": 255
}


,subset,sentences,xml_files,loss,token_acc,precision,recall,f1,gold_entities,pred_entities,correct_entities
0,DrugBank,145,54,4.878257,0.971823,0.891026,0.914474,0.902597,304,312,278
1,MedLine,520,58,5.398599,0.969110,0.615942,0.667539,0.640704,382,414,255


In [36]:
def predict_entities(text, model, tokenizer, id2label, max_length=192):
    model.eval()
    encoding = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True,
        return_tensors="pt",
    )
    offsets = encoding.pop("offset_mapping")[0].tolist()
    encoding = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        path = model.decode(encoding["input_ids"], encoding["attention_mask"])[0]

    token_tags = []
    for tag_id, (start, end) in zip(path, offsets):
        if start == 0 and end == 0:
            continue
        token_tags.append((start, end, id2label[tag_id]))

    spans = []
    current = None
    for start, end, tag in token_tags:
        if tag == "O":
            if current:
                spans.append(current)
                current = None
            continue
        prefix, typ = tag.split("-", 1)
        if prefix == "B" or current is None or current["type"] != typ:
            if current:
                spans.append(current)
            current = {"type": typ, "start": start, "end": end}
        else:
            current["end"] = end
    if current:
        spans.append(current)

    for span in spans:
        span["text"] = text[span["start"]:span["end"]]
    return spans

sample_text = "Aspirin may increase the anticoagulant activities of Warfarin."
predict_entities(sample_text, model, tokenizer, id2label, MAX_LENGTH)

[{'type': 'brand', 'start': 0, 'end': 7, 'text': 'Aspirin'},
 {'type': 'drug', 'start': 53, 'end': 61, 'text': 'Warfarin'}]